# Phase 1.7: Bootstrap Null v2a-RSN Calibration

Run block bootstrap calibration on v2a-RSN real neural recording data using c-GC and c-GC* outputs.

Workflow:
1. Load v2a-RSN c-GC and c-GC* outputs (transitions.csv, summary.json)
2. For each fish and method:
   - Extract depth trajectories (D_p values)
   - Run moving-block bootstrap (block length 50)
   - Compute null T_boot distribution and critical values
   - Extract pointwise null bands for depth-specific thresholding
3. Export results and create depth_bands.png showing observed vs. null envelopes

Output directory: outputs/calibration/v2a/

In [ ]:
from __future__ import annotations

import sys
import json
import logging
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path('.').resolve()
while not (PROJECT_ROOT / 'src' / 'markovianity_diagnostic').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not find project root")

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.bootstrap import bootstrap_global_test

logger.info(f"Project root: {PROJECT_ROOT}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'calibration' / 'v2a'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

V2A_C_GC_DIR = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'c-GC'
V2A_C_GC_STAR_DIR = PROJECT_ROOT / 'outputs' / 'v2a-RSNs' / 'c-GC-star'

B = 50
BLOCK_LENGTH = 50
SEED = 42
CRITICAL_LEVELS = [0.90, 0.95, 0.99]
METHODS_TO_CALIBRATE = ['c-GC', 'c-GC-star']

logger.info(f"Output directory: {OUTPUT_DIR}")
logger.info(f"Bootstrap parameters: B={B}, block_length={BLOCK_LENGTH}, seed={SEED}")
logger.info(f"Critical levels: {CRITICAL_LEVELS}")
logger.info(f"Methods to calibrate: {METHODS_TO_CALIBRATE}")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Bootstrap parameters:")
print(f"  B={B}, block_length={BLOCK_LENGTH}, seed={SEED}")
print(f"  critical_levels={CRITICAL_LEVELS}")
print(f"  Methods: {METHODS_TO_CALIBRATE}")

In [ ]:
# Check for cached outputs
expected_outputs = {
    'bootstrap_results.json': OUTPUT_DIR / 'bootstrap_results.json',
    'bootstrap_summary.csv': OUTPUT_DIR / 'bootstrap_summary.csv',
    'depth_bands.png': OUTPUT_DIR / 'depth_bands.png',
    'manifest.json': OUTPUT_DIR / 'manifest.json',
}

outputs_exist = all(fpath.exists() for fpath in expected_outputs.values())

if outputs_exist:
    print("✅ Outputs already exist - loading cached results")
    print(f"Output directory: {OUTPUT_DIR}")
    for name, path in expected_outputs.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {name} ({size_mb:.2f} MB)")
else:
    print("⚠️ No existing outputs - will run computation")
    print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
if not outputs_exist:
    logger.info("Loading v2a-RSN data...")
    start_load = time.time()
    print("\nLoading v2a-RSN data...")

    c_gc_transitions_path = V2A_C_GC_DIR / 'transitions.csv'
    c_gc_star_transitions_path = V2A_C_GC_STAR_DIR / 'transitions.csv'

    if not c_gc_transitions_path.exists():
        logger.error(f"c-GC transitions not found: {c_gc_transitions_path}")
        raise FileNotFoundError(f"c-GC transitions not found: {c_gc_transitions_path}")
    if not c_gc_star_transitions_path.exists():
        logger.error(f"c-GC-star transitions not found: {c_gc_star_transitions_path}")
        raise FileNotFoundError(f"c-GC-star transitions not found: {c_gc_star_transitions_path}")

    c_gc_transitions_df = pd.read_csv(c_gc_transitions_path)
    c_gc_star_transitions_df = pd.read_csv(c_gc_star_transitions_path)

    logger.info(f"c-GC transitions shape: {c_gc_transitions_df.shape}")
    logger.info(f"c-GC-star transitions shape: {c_gc_star_transitions_df.shape}")
    print(f"c-GC transitions shape: {c_gc_transitions_df.shape}")
    print(f"c-GC-star transitions shape: {c_gc_star_transitions_df.shape}")
    print(f"\nc-GC columns: {c_gc_transitions_df.columns.tolist()}")

    datasets = sorted(c_gc_transitions_df['dataset'].unique())
    logger.info(f"Datasets found: {datasets}")
    print(f"Datasets: {datasets}")

    dataset_mapping = {name: f'fish-{i+1}' for i, name in enumerate(datasets)}
    c_gc_transitions_df['fish'] = c_gc_transitions_df['dataset'].map(dataset_mapping)
    c_gc_star_transitions_df['fish'] = c_gc_star_transitions_df['dataset'].map(dataset_mapping)

    logger.info(f"Mapped to: {sorted(c_gc_transitions_df['fish'].unique())}")
    print(f"\nMapped to: {sorted(c_gc_transitions_df['fish'].unique())}")
    
    load_elapsed = time.time() - start_load
    logger.info(f"Data loading completed in {load_elapsed:.2f}s")
else:
    logger.info("Skipping data loading (cached outputs)")
    print("Skipping data loading (cached outputs)")

In [ ]:
if not outputs_exist:
    logger.info("Extracting depth columns...")
    
    def extract_depth_columns(df: pd.DataFrame) -> dict[int, str]:
        depth_cols = {}
        for col in df.columns:
            if col.startswith('edge_count_p'):
                try:
                    depth = int(col.split('p')[1])
                    depth_cols[depth] = col
                except (ValueError, IndexError):
                    pass
        return depth_cols

    depth_cols_c_gc = extract_depth_columns(c_gc_transitions_df)
    depth_cols_c_gc_star = extract_depth_columns(c_gc_star_transitions_df)

    logger.info(f"c-GC depth columns: {depth_cols_c_gc}")
    logger.info(f"c-GC-star depth columns: {depth_cols_c_gc_star}")
    print(f"c-GC depth columns: {depth_cols_c_gc}")
    print(f"c-GC-star depth columns: {depth_cols_c_gc_star}")

    if set(depth_cols_c_gc.keys()) != set(depth_cols_c_gc_star.keys()):
        logger.warning("Depth columns differ between c-GC and c-GC-star")
        print("Warning: depth columns differ between c-GC and c-GC-star")

    p_values = sorted(depth_cols_c_gc.keys())
    logger.info(f"P-values: {p_values}")
    print(f"\nP-values: {p_values}")
else:
    logger.info("Skipping depth column extraction (cached outputs)")
    print("Skipping depth column extraction (cached outputs)")

In [ ]:
if not outputs_exist:
    logger.info("Starting bootstrap calibration...")
    start_bootstrap = time.time()
    bootstrap_results = {}

    method_data = {
        'c-GC': c_gc_transitions_df,
        'c-GC-star': c_gc_star_transitions_df,
    }

    for method_idx, (method_name, df) in enumerate(method_data.items(), 1):
        logger.info(f"\n[{method_idx}/{len(method_data)}] Method: {method_name}")
        print(f"\n{'='*60}")
        print(f"[{method_idx}/{len(method_data)}] Method: {method_name}")
        print(f"{'='*60}")
        
        method_results = {}
        fish_list = sorted(df['fish'].unique())
        
        for fish_idx, fish in enumerate(fish_list, 1):
            logger.info(f"  [{fish_idx}/{len(fish_list)}] {fish}")
            fish_data = df[df['fish'] == fish].copy()
            
            edge_counts = {}
            for p, col in depth_cols_c_gc.items():
                if col in fish_data.columns:
                    edge_counts[p] = fish_data[col].values.astype(float)
            
            logger.info(f"    Recordings: {len(fish_data)}")
            logger.info(f"    Depth columns: {list(edge_counts.keys())}")
            print(f"\n{fish}:")
            print(f"  Recordings: {len(fish_data)}")
            print(f"  Depth columns: {list(edge_counts.keys())}")
            
            edge_count_cols = [depth_cols_c_gc[p] for p in sorted(edge_counts.keys())]
            X = fish_data[edge_count_cols].values.astype(float)
            
            logger.info(f"    X shape for bootstrap: {X.shape}")
            print(f"  X shape for bootstrap: {X.shape}")
            
            def edge_count_analyzer(X_boot: np.ndarray, p_values: list[int]) -> dict:
                result = {'D_p': {}}
                for col_idx, p in enumerate(sorted(edge_counts.keys())):
                    result['D_p'][p] = X_boot[:, col_idx].mean()
                result['T_obs'] = max(result['D_p'].values()) if result['D_p'] else 0.0
                return result
            
            fish_start = time.time()
            result = bootstrap_global_test(
                X,
                analyze_fn=edge_count_analyzer,
                p_values=list(sorted(edge_counts.keys())),
                p0=1,
                B=B,
                block_length=BLOCK_LENGTH,
                seed=SEED + hash(f"{method_name}_{fish}") % 1000,
            )
            fish_elapsed = time.time() - fish_start
            
            T_obs = result.get('T_obs', 0.0)
            T_boot = result.get('T_boot', [])
            
            T_boot_array = np.asarray(T_boot)
            critical_values = {}
            for level in CRITICAL_LEVELS:
                critical_values[level] = float(np.quantile(T_boot_array, level))
            
            p_value = float((1 + np.sum(T_boot_array >= T_obs)) / (B + 1))
            
            method_results[fish] = {
                'T_obs': T_obs,
                'T_boot': T_boot,
                'critical_values': critical_values,
                'p_value': p_value,
                'edge_counts': edge_counts,
            }
            
            logger.info(f"    T_obs: {T_obs:.4f}, p-value: {p_value:.4f} (elapsed: {fish_elapsed:.2f}s)")
            print(f"  T_obs: {T_obs:.4f}")
            print(f"  p-value: {p_value:.4f}")
            print(f"  Critical values: {critical_values}")
            print(f"  Elapsed: {fish_elapsed:.2f}s")
        
        bootstrap_results[method_name] = method_results

    total_bootstrap = time.time() - start_bootstrap
    logger.info(f"\nBootstrap calibration complete (total: {total_bootstrap:.2f}s)")
    print("\n✓ Bootstrap calibration complete")
else:
    logger.info("Skipping bootstrap computation (cached outputs)")
    print("Skipping bootstrap computation (cached outputs)")

In [ ]:
# Load cached bootstrap results if they exist
if outputs_exist:
    bootstrap_results_path = expected_outputs['bootstrap_results.json']
    summary_path = expected_outputs['bootstrap_summary.csv']
    manifest_path = expected_outputs['manifest.json']
    
    with open(bootstrap_results_path, 'r') as f:
        bootstrap_results_json = json.load(f)
    
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    # Reconstruct bootstrap_results dict from cached JSON
    bootstrap_results = {}
    for method_name, method_data in bootstrap_results_json.items():
        bootstrap_results[method_name] = {}
        for fish, data in method_data.items():
            bootstrap_results[method_name][fish] = {
                'T_obs': data['T_obs'],
                'T_boot': data['T_boot'],
                'critical_values': {float(k): v for k, v in data['critical_values'].items()},
                'p_value': data['p_value'],
                'edge_counts': {},
            }
    
    summary_df = pd.read_csv(summary_path)
    datasets = manifest.get('n_fish', 2)
    p_values = manifest.get('p_values', [])
    
    print(f"Loaded cached bootstrap results for {len(bootstrap_results)} methods")
    print(f"  Methods: {list(bootstrap_results.keys())}")
    print(f"  P-values: {p_values}")

In [ ]:
logger.info("Exporting bootstrap results...")

bootstrap_results_json = {}
for method_name, method_data in bootstrap_results.items():
    bootstrap_results_json[method_name] = {}
    for fish, data in method_data.items():
        bootstrap_results_json[method_name][fish] = {
            'T_obs': float(data['T_obs']),
            'T_boot': [float(x) for x in data['T_boot']],
            'critical_values': {float(k): float(v) for k, v in data['critical_values'].items()},
            'p_value': float(data['p_value']),
        }

bootstrap_results_path = OUTPUT_DIR / 'bootstrap_results.json'
with open(bootstrap_results_path, 'w') as f:
    json.dump(bootstrap_results_json, f, indent=2)

logger.info(f"Exported: {bootstrap_results_path}")
print(f"Exported: {bootstrap_results_path}")

In [ ]:
logger.info("Exporting summary CSV...")

summary_data = []
for method_name, method_data in bootstrap_results.items():
    for fish, data in method_data.items():
        row = {
            'fish': fish,
            'method': method_name,
            'T_obs': float(data['T_obs']),
            'p_value': float(data['p_value']),
            'critical_90': data['critical_values'][0.90],
            'critical_95': data['critical_values'][0.95],
            'critical_99': data['critical_values'][0.99],
            'B': B,
        }
        summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_path = OUTPUT_DIR / 'bootstrap_summary.csv'
summary_df.to_csv(summary_path, index=False)

logger.info(f"Exported: {summary_path}")
logger.info(f"Summary: {len(summary_df)} rows")
print(f"\nExported: {summary_path}")
print(f"Summary (n={len(summary_df)} rows):")
print(summary_df.to_string())

In [ ]:
logger.info("Generating depth band plots...")
start_plot = time.time()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

plot_idx = 0
for method_name in ['c-GC', 'c-GC-star']:
    method_data_local = bootstrap_results[method_name]
    
    for fish in sorted(method_data_local.keys()):
        logger.info(f"  Plotting {fish} - {method_name}")
        ax = axes[plot_idx]
        data = method_data_local[fish]
        T_boot = np.asarray(data['T_boot'])
        T_obs = data['T_obs']
        
        ax.hist(T_boot, bins=15, alpha=0.6, color='steelblue', edgecolor='black', label='Bootstrap null')
        ax.axvline(T_obs, color='red', linestyle='--', linewidth=2, label=f'Observed T={T_obs:.3f}')
        
        cv_95 = data['critical_values'][0.95]
        ax.axvline(cv_95, color='orange', linestyle=':', linewidth=2, alpha=0.7, label=f'Critical 95%={cv_95:.3f}')
        
        ax.set_xlabel('Test Statistic')
        ax.set_ylabel('Frequency')
        ax.set_title(f"{fish} - {method_name}")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        
        plot_idx += 1

for idx in range(plot_idx, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plot_path = OUTPUT_DIR / 'depth_bands.png'
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plot_elapsed = time.time() - start_plot
logger.info(f"Exported: {plot_path} (elapsed: {plot_elapsed:.2f}s)")
print(f"Exported: {plot_path}")
plt.close()

In [ ]:
logger.info("Creating manifest...")

def _get_git_commit() -> str:
    try:
        import subprocess
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=5,
            cwd=PROJECT_ROOT,
        )
        return result.stdout.strip() if result.returncode == 0 else "unknown"
    except Exception:
        return "unknown"

manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "git_commit": _get_git_commit(),
    "analysis": "bootstrap_null_v2a",
    "input_paths": [
        str(c_gc_transitions_path),
        str(c_gc_star_transitions_path),
    ],
    "output_paths": [
        str(bootstrap_results_path),
        str(summary_path),
        str(plot_path),
    ],
    "methods": METHODS_TO_CALIBRATE,
    "method_params": {
        "B": B,
        "block_length": BLOCK_LENGTH,
        "critical_levels": CRITICAL_LEVELS,
        "p0": 1,
    },
    "n_fish": len(datasets),
    "p_values": p_values,
    "random_seed": SEED,
    "software_versions": {
        "python": f"{sys.version.split()[0]}",
        "numpy": f"{np.__version__}",
        "pandas": f"{pd.__version__}",
        "matplotlib": f"{plt.matplotlib.__version__}",
    },
}

manifest_path = OUTPUT_DIR / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

logger.info(f"Exported: {manifest_path}")
print(f"Exported: {manifest_path}")
print(json.dumps(manifest, indent=2))

In [ ]:
logger.info("Verifying outputs...")
print("\nVerification:")
expected_files = [
    bootstrap_results_path,
    summary_path,
    plot_path,
    manifest_path,
]

all_exist = True
for fpath in expected_files:
    exists = fpath.exists()
    status = '✓' if exists else '✗'
    size = fpath.stat().st_size if exists else 'N/A'
    print(f"{status} {fpath.name} (size: {size} bytes)")
    logger.info(f"{status} {fpath.name}")
    all_exist = all_exist and exists

logger.info(f"CSV verification: {len(summary_df)} rows")
print(f"\nCSV verification:")
print(f"  bootstrap_summary.csv: {len(summary_df)} rows (expected {2 * len(datasets)})")
assert len(summary_df) == 2 * len(datasets), "CSV row count mismatch"

if all_exist:
    logger.info("✓ All outputs verified successfully")
    print("\n✓ All outputs verified successfully")
else:
    logger.error("✗ Some outputs are missing")
    print("\n✗ Some outputs are missing")